In [1]:
from tensorflow.keras.models import load_model
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import pickle
from html.parser import HTMLParser
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


Download required NLTK resources

In [2]:
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
with open("vectorizer1.pkl", "rb") as f:
    vectorizer1 = pickle.load(f)

with open("vectorizer2.pkl", "rb") as f:
    vectorizer2 = pickle.load(f)

with open("vectorizer3.pkl", "rb") as f:
    vectorizer3 = pickle.load(f)


In [4]:
vectorizer1 = pickle.load(open("vectorizer1.pkl", "rb"))
vectorizer2 = pickle.load(open("vectorizer2.pkl", "rb"))
vectorizer3 = pickle.load(open("vectorizer3.pkl", "rb"))

In [5]:
model1 = load_model("model_config1.keras")
model2 = load_model("model_config2.keras")
model3 = load_model("model_config3.keras")

In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [7]:
class HTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.fed = []

    def handle_data(self, d):
        self.fed.append(d)

    def get_data(self):
        return ''.join(self.fed)


def remove_html(text):
    s = HTMLStripper()
    s.feed(text)
    return s.get_data()


def remove_emoji(text):
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub('', text)


def precleaning(text):
    text = remove_html(text)
    text = remove_emoji(text)
    return text

def cleaning_conf1(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

def cleaning_conf2(text):

    text = cleaning_conf1(text)
    filtered_words = []

    for word in text.split():
        if word not in stop_words:
            filtered_words.append(word)

    return " ".join(filtered_words)

def cleaning_conf3(text):
    text = cleaning_conf1(text)
    lemmatized_words = []

    for word in text.split():
        lemmatized_words.append(lemmatizer.lemmatize(word))

    return " ".join(lemmatized_words)


In [16]:
import sys
!{sys.executable} -m pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 10.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 12.4 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]



In [9]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
tweet_box = widgets.Text(
    value='',
    placeholder='Type a tweet here...',
    description='Tweet:',
    layout=widgets.Layout(width='600px')
)

predict_button = widgets.Button(
    description="Classify Tweet",
    button_style='success'
)

output = widgets.Output()

In [11]:
def predict_tweet(b):

    with output:
        output.clear_output()

        tweet = tweet_box.value

        preprocessed_tweet = precleaning(tweet)

        cleaned_tweet_conf1 = cleaning_conf1(preprocessed_tweet)
        cleaned_tweet_conf2 = cleaning_conf2(preprocessed_tweet)
        cleaned_tweet_conf3 = cleaning_conf3(preprocessed_tweet)

        vector1 = vectorizer1.transform([cleaned_tweet_conf1])
        vector2 = vectorizer2.transform([cleaned_tweet_conf2])
        vector3 = vectorizer3.transform([cleaned_tweet_conf3])

        pred1 = model1.predict(vector1)[0][0]
        pred2 = model2.predict(vector2)[0][0]
        pred3 = model3.predict(vector3)[0][0]

        def label(prob):
            return "Disaster" if prob > 0.5 else "Not Disaster"

        print("Tweet:", tweet)
        print("\n--- Predictions ---")

        print(f"Model 1 (Config 1): {label(pred1)} | confidence: {pred1:.3f}")
        print(f"Model 2 (Config 2): {label(pred2)} | confidence: {pred2:.3f}")
        print(f"Model 3 (Config 3): {label(pred3)} | confidence: {pred3:.3f}")

In [12]:
predict_button.on_click(predict_tweet)

In [13]:
display(tweet_box, predict_button, output)

Text(value='', description='Tweet:', layout=Layout(width='600px'), placeholder='Type a tweet here...')

Button(button_style='success', description='Classify Tweet', style=ButtonStyle())

Output()